# Instacart Grocery Recommendations 
This notebook contains the code for our exam project in 02807 Computational Tools for Data Science.  

The goal of our project is to implement and evaluate multiple recommenders for groceries. We are working with the [Instacart Online Grocery Basket Analysis Dataset](https://www.kaggle.com/datasets/yasserh/instacart-online-grocery-basket-analysis-dataset/data?select=order_products__prior.csv), which includes approximately 50,000 products, 200,000 users, and 3.4 million orders. 

We are utilizing the following algorithms for our recommenders:
- KMeans 
- Apriori
- Collaborative Filtering (CF)

Our project introduces the following recommenders:
- Top-n recommender
- Apriori Recommender
- CF on full user-item matrix (CF_Full)
- CF on user-aisle matrix (CF_A)
- CF with user clusters and all items (CF_C)
- CF with user clusters and aisles (CF_AC)

Contributors:
- Andreas Kruse Svenningsen (s253844)
- Frederik Winther Bæk (s214618)
- Georgios Loulakis (s252920)
- Sebastian Nygaard Wærling (s254120)

To run our you must initialize an environment containing all the packages specified in the `requirements.txt` file. This can be done with homebrew running the following commands:

In [ ]:
# imports
import os
import sys 

import kagglehub
import pandas as pd
import shutil
from tqdm import tqdm

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), ".")))
from config import *

The following code blocks follow the execution order of scripts specified in the `pipeline.py` script. Some of the recommenders require an substantial amount of memory (RAM) and have been executed using the DTU HPC. We have decided to exclude those part from this notebook. It will be stated when code has been excluded.  

## 1. Download Dataset
The following functions are used to download the Instacart dataset from Kaggle.

In [1]:
def download_dataset() -> str:
    # Download latest version
    path = kagglehub.dataset_download("yasserh/instacart-online-grocery-basket-analysis-dataset")

    print("Path to dataset files:", path)
    return path

def move_dataset_from_cache_to_folder(path_to_cache: str, path_to_folder: str) -> None:
    shutil.copytree(path_to_cache, path_to_folder, dirs_exist_ok=True)
    shutil.rmtree(path_to_folder / "data", ignore_errors=True)

def convert_to_parquet() -> None:

    for file in tqdm(os.listdir(DATA_RAW_DIR)):
        file_name, file_extension = file.split(".")
        file_extension = "."+(file_extension)
        pd.read_csv(DATA_RAW_DIR / (file_name + file_extension)).to_parquet(DATA_CLEANED_DIR / (file_name + ".pq"))

Now we can use the functions to download the dataset:

In [ ]:
path_to_cache = download_dataset()
move_dataset_from_cache_to_folder(path_to_cache=path_to_cache, path_to_folder=DATA_RAW_DIR)
convert_to_parquet()

## 2. Data Split
The following code block saves the raw data to parquet files and divides the orders into a train, validation, and test set. The three sets are also saved as parquet files named `order_products__train`.  

In [ ]:
# load data
orders_df = pd.read_csv(ORDERS_PATH_CSV)
op_prior = pd.read_csv(ORDER_PRODUCTS__PRIOR_PATH_CSV)
op_train = pd.read_csv(ORDER_PRODUCTS__TRAIN_PATH_CSV)

# remove test orders
orders_df = orders_df[orders_df["eval_set"] != "test"]

# sorting to ensure correct ordering
orders_df = orders_df.sort_values(["user_id", "order_number"])

# helper column counting number of orders per user
orders_df["n_orders"] = orders_df.groupby("user_id")["order_number"].transform("max")

# assign split labels (order 1,...,n-2: train, order n-1: val, order n: test)
orders_df["eval_set_new"] = "train"
orders_df.loc[orders_df["order_number"] == orders_df["n_orders"], "eval_set_new"] = "test"
orders_df.loc[orders_df["order_number"] == orders_df["n_orders"] - 1, "eval_set_new"] = "val"

# drop n_orders and make eval_set_new the new eval_set column
orders_df["eval_set"] = orders_df["eval_set_new"]
orders_df = orders_df.drop(columns=["eval_set_new", "n_orders"])

# save orders_df to parquet
orders_df.to_parquet(ORDERS_PATH)

# concatenate order_products data
op_combined = pd.concat([op_prior, op_train])

# order_ids in each split
train_orders = orders_df[orders_df["eval_set"]=="train"]["order_id"]
val_orders = orders_df[orders_df["eval_set"]=="val"]["order_id"]
test_orders = orders_df[orders_df["eval_set"]=="test"]["order_id"]

# order products for each split
op_train_new = op_combined[op_combined["order_id"].isin(train_orders)]
op_val_new = op_combined[op_combined["order_id"].isin(val_orders)]
op_test_new = op_combined[op_combined["order_id"].isin(test_orders)]

# saving to parquet
op_train_new.to_parquet(ORDER_PRODUCTS__TRAIN_PATH)
op_val_new.to_parquet(ORDER_PRODUCTS__VAL_PATH)
op_test_new.to_parquet(ORDER_PRODUCTS__TEST_PATH)

## 3. Calculate Rating

## 4. Clustering

## 5. Baseline Recommender

## 6. Collaborative Filtering

## 7. Apriori